In [1]:
import scipy as sp
import numpy as np
import copy
import numpy.random as rnd

The gradient (and loss) as a function :

In [2]:
def GradERM(X, y, w, v, sigmafunc, Dsigmafunc, LambdaRegularization):
    zw = np.matmul(X, w)
    zv = np.matmul(X, v)
    GradwTotal = np.matmul(np.transpose(X), - np.divide(y - zw,sigmafunc(zv))) + LambdaRegularization*w
    GradvTotal = (np.matmul(np.transpose(X), np.multiply((np.divide(1,(2*sigmafunc(zv))) - np.divide(np.power((y - zw),2),(2*np.power(sigmafunc(zv),2)))), Dsigmafunc(zv))) 
                  + LambdaRegularization*v)
    return(np.stack((GradwTotal, GradvTotal), axis = 1))

def LossERM(X, y, w, v, sigmafunc, LambdaRegularization):
    zw = np.matmul(X, w)
    zv = np.matmul(X, v)
    return( np.sum(np.divide(np.power((y - zw),2),(2*sigmafunc(zv))) + np.log(sigmafunc(zv))/2, 0) + LambdaRegularization*(np.dot(w,w) + np.dot(v,v))/2 )

The GD step

In [3]:
def GDStepERM(X, y, wv, sigmafunc, Dsigmafunc, LambdaRegularization, LearningRate):
    wv -= LearningRate*GradERM(X, y, wv[:,0], wv[:,1], sigmafunc, Dsigmafunc, LambdaRegularization)
    return(LossERM(X, y, wv[:,0], wv[:,1], sigmafunc, LambdaRegularization))

Full GD function

In [4]:
def GDERM(X, y, wv, sigmafunc, Dsigmafunc, LambdaRegularization = 1, LearningRate = 0.02, MaxIter = 1e4, EpsConvergence = 1e-7, Verbose = True, VerboseRate = 100):
    Conv = 1
    NIter = 0
    Losses = [LossERM(X, y, wv[:,0], wv[:,1], sigmafunc, LambdaRegularization)]
    if(Verbose):
        print("Iteration %s" % NIter)
        print("Current loss %s" % Losses[NIter])
    while((NIter < MaxIter) and (Conv > EpsConvergence)):
        Losses.append(GDStepERM(X, y, wv, sigmafunc, Dsigmafunc, LambdaRegularization, LearningRate))
        NIter = NIter + 1
        Conv = np.abs(Losses[NIter] - Losses[NIter-1])/np.abs(Losses[NIter])
        if(Verbose and NIter%VerboseRate == 0):
            print("Iteration %s" % NIter)
            print("Current loss %s" % Losses[NIter])
            print("Current convergence criterion %s" % Conv)
    if(Verbose):
        print("Iteration %s" % NIter)
        print("Current loss %s" % Losses[NIter])
        print("Current convergence criterion %s" % Conv)
    return(np.array(Losses))
        

Sigma Functions for $\sigma(z_v) = z_v^2$

In [5]:
def sigmaSquare(zv):
    return(np.power(zv,2))

def DsigmaSquare(zv):
    return(2*zv)

Sigma Functions for $\sigma(z_v) = \log(1 + e^{z_v})$

In [6]:
def sigmaSoftplus(zv):
    return(np.log(1 + np.exp(zv)))

def DsigmaSoftplus(zv):
    return(np.divide(1, 1 + np.exp(-zv)))

Main variables 

In [7]:
d = 2000
LearningRate = 0.002
Reps = 20

Runner

In [ ]:
for alpha in [1.0, 3.0, 5.0, 7.0, 10.0]:
    for LambdaRegularization in [1.0, 2.0, 5.0]:
        m = np.zeros(2)
        m2 = np.zeros(2)
        q = np.zeros(2)
        q2 = np.zeros(2)
        for _ in range(Reps):
            M = int(alpha*d)
            X = rnd.normal(0, 1/np.sqrt(d), size = (M, d))
            wvTrue = rnd.normal(0, 1, size = (d, 2))
            wvLearned = rnd.normal(0, 1, size = (d, 2))
            yTrue = np.einsum("ij,j->i", X, wvTrue[:,0])
            yNoise = yTrue + rnd.normal(0, np.sqrt(sigmaSoftplus(np.matmul(X, wvTrue[:,1]))))
            GDERM(X, yNoise, wvLearned, sigmaSoftplus, DsigmaSoftplus, LearningRate=LearningRate, LambdaRegularization=LambdaRegularization, MaxIter = 100000, VerboseRate = 5000, EpsConvergence=1e-6)
            m += np.array([np.dot(wvLearned[:,0], wvTrue[:,0])/d, np.dot(wvLearned[:,1], wvTrue[:,1])/d ])
            m2 += np.square(np.array([np.dot(wvLearned[:,0], wvTrue[:,0])/d, np.dot(wvLearned[:,1], wvTrue[:,1])/d ]))
            q += np.array([np.dot(wvLearned[:,0], wvLearned[:,0])/d, np.dot(wvLearned[:,1], wvLearned[:,1])/d ])
            q2 += np.square(np.array([np.dot(wvLearned[:,0], wvLearned[:,0])/d, np.dot(wvLearned[:,1], wvLearned[:,1])/d ]))
        vm = (m2 - np.square(m)/Reps)/(Reps - 1)
        m /= Reps
        vq = (q2 - np.square(q)/Reps)/(Reps - 1)
        q /= Reps
        np.savetxt(f"q_alpha_{alpha}_Lambda_{LambdaRegularization}_d_{d}_GD_Neo.txt", q, fmt="%.6f")
        np.savetxt(f"varq_alpha_{alpha}_Lambda_{LambdaRegularization}_d_{d}_GD_Neo.txt", vq, fmt="%.6f")
        np.savetxt(f"m_alpha_{alpha}_Lambda_{LambdaRegularization}_d_{d}_GD_Neo.txt", m, fmt="%.6f")
        np.savetxt(f"varm_alpha_{alpha}_Lambda_{LambdaRegularization}_d_{d}_GD_Neo.txt", vm, fmt="%.6f")